<a href="https://colab.research.google.com/github/huyle3/Data_eagles_wharton_comp/blob/main/General_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
# import tensorflow as tf
# !git clone https://github.com/huyle3/Data_eagles_wharton_comp.git
pure_dataset = pd.read_csv("Data/whl_2025.csv")

In [12]:
cols_to_drop = [ #Columns with object datatype besides gameid
    "went_ot", "home_off_line",
    "home_def_pairing", "away_off_line",
    "away_def_pairing", "home_goalie",
    "away_goalie", "record_id"
]
df_only_num = pure_dataset.drop(columns=cols_to_drop)



#reduce 10 records per game to 1; add from the game
df_sum = (
      df_only_num
      .groupby("game_id", as_index=False)
      .agg({
          "home_team": "first",
          "away_team": "first",
          **{
              col: "sum"
              for col in df_only_num.select_dtypes("number").columns
              if col != "id"
          }
      })
  )
#new column; home team wins = 1, away team wins = 0, draw = =1

df_sum["result"] = np.select(
    [
        df_sum["home_goals"] > df_sum["away_goals"],
        df_sum["home_goals"] <= df_sum["away_goals"],
    ],
    [1, 0]
)
df_sum.head()




,game_id,home_team,away_team,toi,home_assists,home_shots,home_xg,home_max_xg,home_goals,away_assists,away_shots,away_xg,away_max_xg,away_goals,home_penalties_committed,home_penalty_minutes,away_penalties_committed,away_penalty_minutes,result
0,game_1,thailand,pakistan,3599.99,2,21,2.8231,1.3905,1,6,24,2.7516,1.2546,3,8,16,6,12,0
1,game_10,switzerland,kazakhstan,3600.01,7,20,1.9254,1.2758,4,4,30,3.3189,1.2719,3,10,20,0,0,1
2,game_100,serbia,rwanda,3600.00,7,30,3.6712,1.7836,4,10,27,3.0240,1.2909,5,8,16,10,20,0
3,game_1000,brazil,netherlands,3600.03,6,32,3.5905,1.5239,5,0,27,2.5261,1.1340,0,10,20,6,12,1
4,game_1001,india,morocco,3599.99,4,32,3.4592,1.8355,2,6,29,3.7658,1.2034,3,8,16,8,16,0


In [ ]:
def reform_ds(df):
  home_stat_cols = [c for c in df.columns if c.startswith("home_") and c != "home_team"]
  away_stat_cols = [c for c in df.columns if c.startswith("away_") and c != "away_team"]

  home_long = df[["game_id", "home_team"] + home_stat_cols].copy()
  home_long.columns = ["game_id", "team"] + [
      c.replace("home_", "") for c in home_stat_cols
  ]

  away_long = df[["game_id", "away_team"] + away_stat_cols].copy()
  away_long.columns = ["game_id", "team"] + [
      c.replace("away_", "") for c in away_stat_cols
  ]

  long_df = pd.concat([home_long, away_long], ignore_index=True)

  stat_cols = [c for c in long_df.columns if c not in ["game_id", "team"]]

  team_sum = long_df.groupby("team")[stat_cols].transform("sum")
  team_count = long_df.groupby("team")[stat_cols].transform("count")

  avg_excl = (team_sum - long_df[stat_cols]) / (team_count - 1)

  n = len(df)

  home_avg = avg_excl.iloc[:n].add_prefix("home_team_avg_")
  away_avg = avg_excl.iloc[n:].add_prefix("away_team_avg_")

  df_final = pd.concat(
      [df.reset_index(drop=True),
      home_avg.reset_index(drop=True),
      away_avg.reset_index(drop=True)],
      axis=1
  )
  return df_final
new_ds = reform_ds(df_sum)
new_ds.head()

,game_id,home_team,away_team,toi,home_assists,home_shots,home_xg,home_max_xg,home_goals,away_assists,...,home_team_avg_goals,home_team_avg_penalties_committed,home_team_avg_penalty_minutes,away_team_avg_assists,away_team_avg_shots,away_team_avg_xg,away_team_avg_max_xg,away_team_avg_goals,away_team_avg_penalties_committed,away_team_avg_penalty_minutes
0,game_1,thailand,pakistan,3599.99,2,21,2.8231,1.3905,1,6,...,3.617284,7.000000,14.308642,5.012346,30.037037,3.480886,1.402268,3.209877,8.024691,16.296296
1,game_10,switzerland,kazakhstan,3600.01,7,20,1.9254,1.2758,4,4,...,2.432099,7.308642,14.913580,3.728395,24.148148,2.549215,1.106674,2.345679,5.172840,10.543210
2,game_100,serbia,rwanda,3600.00,7,30,3.6712,1.7836,4,10,...,3.271605,6.358025,12.925926,3.962963,25.913580,2.725746,1.250480,2.432099,8.148148,16.617284
3,game_1000,brazil,netherlands,3600.03,6,32,3.5905,1.5239,5,0,...,3.345679,6.950617,14.185185,4.839506,25.888889,2.962623,1.251820,3.000000,5.555556,11.333333
4,game_1001,india,morocco,3599.99,4,32,3.4592,1.8355,2,6,...,2.703704,5.555556,11.308642,4.839506,28.777778,3.033796,1.261572,3.037037,5.950617,12.024691


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score




In [15]:
rf = RandomForestClassifier()
def make_predictions(data, predictors):
  X_train, X_test, y_train, y_test = train_test_split(data[predictors], data["result"], test_size=0.2, random_state=42)
  rf.fit(X_train, y_train)
  preds = rf.predict(X_test)
  combined = pd.DataFrame(dict(actual=y_test, predicted=preds), index=X_test.index)
  precision = precision_score(y_test, preds)
  accuracy = accuracy_score(y_test, preds)
  return combined, precision, accuracy

In [16]:
predictors = [col for col in new_ds.columns if "avg" in col]
combined, precision, accuracy = make_predictions(new_ds, predictors)


In [17]:
combined = combined.merge(new_ds[["home_team", "away_team", "result"]], left_index=True, right_index=True)

In [18]:
combined

,actual,predicted,home_team,away_team,result
1027,0,0,kazakhstan,france,0
367,1,1,morocco,vietnam,1
549,0,1,netherlands,france,0
282,1,1,pakistan,morocco,1
678,1,1,ethiopia,philippines,1
...,...,...,...,...,...
261,1,1,panama,germany,1
846,1,1,usa,singapore,1
209,0,0,uk,new_zealand,0
309,0,0,uk,iceland,0


In [19]:
accuracy

0.7414448669201521

In [20]:
precision

0.695906432748538

In [21]:
# Save as CSV
new_ds.to_csv('processed_data.csv', index=False) # Use index=False if you don't need the index in the file
